# `ping_local_host.sh` $-$ Bash ICMP Home Network Host Discovery Prototype

This Bash script is a minimal host discovery tool designed for learning and validation purposes.

---

## v0

### What the script does

- Accepts a CIDR block as a command-line argument (restricted to `/24`, e.g. `192.168.1.0/24`)
- Parses the first three octets of the network address
- Iterates through host addresses `.1` to `.254` (although own host IP could be omitted, this was deemed excessive code for little gain in this example script)
- Sends a single ICMP echo request (`ping -c 1`) to each IP
- Suppresses command output and checks the exit status
- Records and prints IP addresses that respond
- Reports the total number of responsive hosts at the end

### Design Constraints (Intentional)

- Only supports `/24` networks
- No concurrency or optimisation
- No external dependencies (e.g. `nmap`, `ipcalc`)
- Uses standard macOS-compatible `ping`
- Prioritises correctness and architectural clarity over performance

### What this validates

- Basic CIDR parsing strategy
- ICMP reachability behaviour on the local network
- Timeout behaviour and reliability of single-probe detection
- Privilege requirements (no root required for ICMP)
- Structured output for later Python replication

This script represents the first scaffold in a staged development process before re-implementing the logic more robustly in Python using the `ipaddress` module and structured return types.

---

## Improvements in v1 $-$ Input and Output Handling 

#### Prevented Subshell Array Loss

When piping into `while`, Bash runs the loop in a subshell.  
This meant the `RESPONSIVE` array would not persist outside the loop.

Using process substitution keeps the loop in the main shell, so:

- `RESPONSIVE+=()` now works correctly
- Host count is accurate


#### Added Proper CIDR Validation

The script now validates CIDR input using Python’s `ipaddress` module:

    ipaddress.ip_network(CIDR, strict=False)

This prevents malformed input such as:

    999.999.999.0/24


#### Supports Arbitrary CIDR Prefixes

Previously only `/24` was supported.

Now any valid prefix works:

- `/30`
- `/24`
- `/16`
- etc.

Host expansion is handled safely by (and which automatically excludes network address and broadcast address):

    net.hosts()


#### Cleaner Exit Handling

CIDR validation now exits cleanly with an error message if input is invalid.


## Architectural Improvement

The script now separates responsibilities:

- **Bash** → orchestration and control flow
- **Python (`ipaddress`)** → correct subnet math

This reduces fragile manual IP parsing and eliminates off-by-one subnet errors.


## Practical Usability Improvements

- `stdout` and `stderr` handled so that piping to an output records IP addresses only.
    - e.g., `ping_local_hosts.sh 192.168.1.0/24 > output.txt`

    - IPs can still be printed in the terminal (while piping to a file) using `tee`:
        - `ping_local_hosts.sh 192.168.1.0/24 | tee output.txt`

- Interrupt enabled with `Ctrl+C` using `trap`.


## Still Intentionally Minimal

The script still:

- Uses ICMP only
- Has no concurrency
- Has no rate limiting
- Does not attempt ARP discovery
- Does not detect hosts that block ICMP

Correctness and clarity remain the priority at this stage.

---

## Improvements in v2 $—$ Timeout Control and Parallelisation

Two major performance improvements were introduced in this version.

#### 1. Configurable ICMP Timeout

The script now accepts an optional timeout parameter (milliseconds):

    ./ping_local_hosts.sh <CIDR> [timeout_ms] [max_jobs]

Default:
- `timeout_ms = 500`

Previously, each host probe waited up to 1000ms.
Reducing the timeout significantly decreases worst-case runtime.

This makes scan duration approximately:

    runtime ≈ (number_of_hosts × timeout) / concurrency

Trade-off:
- Lower timeout → faster scans
- But increased risk of false negatives on slower or rate-limited devices

This introduces an explicit performance vs reliability control.

#### 2. Controlled Parallelisation

The script now launches multiple `ping` processes concurrently using background jobs.

A second optional parameter controls maximum concurrency:

    max_jobs = 10 (default)

Concurrency is capped using:

    while [ "$(jobs -r | wc -l)" -ge "$MAX_JOBS" ]; do
        sleep 0.05
    done

This ensures:

- No uncontrolled process explosion
- Predictable resource usage
- Stable system behaviour

Performance improvement is substantial:

Sequential scanning:
    ~254 seconds (1s timeout on /24)

Parallel scanning (e.g. 20 jobs, 300ms timeout):
    ~4 seconds

## Architectural Impact

The script now introduces two real-world scanner engineering concepts:

- Probe timing sensitivity
- Controlled concurrency

This transforms the tool from a simple sequential prototype into a lightweight parallel network scanner while still maintaining clarity and minimal dependencies.

Further improvements (future phases) may include:
- ARP-based local discovery
- Rate control and jitter (e.g., randomisation of hosts, non-regular probe intervals, etc.)
- Structured output formats
- Python-native async implementation

---

## Improvements in v3  $—$ Cross-Platform Enhancements — macOS / Linux Usability Alignment

This version introduces explicit OS detection and timeout normalisation to ensure equivalent behaviour between macOS and Linux systems.

#### 1. Automatic OS Detection

The script now detects the operating system using:

    uname

It maps:

- `Darwin` → macOS  
- `Linux` → Linux (e.g. Kali, Ubuntu, Debian)

A status line is printed to `stderr`:

    Detected OS: macOS
    Detected OS: Linux

This improves transparency and debugging clarity when running on different environments.

#### 3. Normalised Timeout Semantics

The `ping` command uses different timeout units across platforms:

| OS      | `ping -W` expects |
|----------|-------------------|
| macOS    | milliseconds      |
| Linux    | seconds (can be fractional) |

To provide a consistent user interface:

- The script always accepts `timeout_ms` in **milliseconds**
- On macOS → value is passed directly to `ping`
- On Linux → value is converted internally to seconds using `awk`

Example:

User input:
    timeout_ms = 300

macOS call:
    ping -W 300

Linux call:
    ping -W 0.300

This prevents silent 500-second timeouts on Linux and ensures equivalent scan behaviour across systems.

#### 3. Consistent User Experience

Users now interact with the script identically on both platforms:

    ./ping_local_hosts.sh <CIDR> [timeout_ms] [max_jobs]

No OS-specific flags are required from the user.

Internally, the script adapts to the host environment while preserving:

- Millisecond-level timing control
- Controlled concurrency
- Clean stdout/stderr separation
- Identical functional behaviour

## Architectural Outcome

These changes eliminate a major portability trap caused by identical flag names (`-W`) with different unit semantics.

The tool now behaves consistently across:

- macOS (BSD ping)
- Linux distributions (iputils ping)

This moves the script from a platform-dependent prototype to a cross-platform scanning utility with predictable timing behaviour.

---